## Sentiment Prediction Demo

This notebook demonstrates how the trained transformer model
can be used to classify new customer reviews.

The notebook loads the optimized DistilBERT model and performs
sentiment prediction on user-provided text input.

Possible sentiment classes:

• Negative  
• Neutral  
• Positive

In [2]:
#Iport required libraries
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

#load optimized transformer model
model_path = "../models/distilbert_3class_v2_optimized"

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path)

#Use GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model.to(device)
model.eval()

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


In [3]:
#Mapping from numeric labels to sentiment classes
labels = ["Negative", "Neutral", "Positive"]

def predict_sentiment(text):
    """
    Predict sentiment of a customer review using the trained model.

    Parameters
    
    text : str
        Customer review text.

    Returns
    
    tuple[str, float]
        Predicted sentiment label and confidence score.
    """
    #Tokenize input text
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )
    #Move tensors to GPU if available
    inputs = {k: v.to(device) for k, v in inputs.items()}

    #Perform inference
    with torch.no_grad():
        outputs = model(**inputs)

    #Convert logits to probabilities
    probs = torch.nn.functional.softmax(outputs.logits, dim=1)
    prediction = torch.argmax(probs).item()

    return labels[prediction], probs[0][prediction].item()

In [4]:
#Example preditioon

text = input("Enter a review: ")
sentiment, confidence = predict_sentiment(text)

sentiment, confidence = predict_sentiment(text)

print("Review:", text)
print("Sentiment:", sentiment)
print("Confidence:", round(confidence, 3))

Enter a review:  Not to bad for the price but expected a little better. Will do the job


Review: Not to bad for the price but expected a little better. Will do the job
Sentiment: Neutral
Confidence: 0.945
